In [ ]:
import pandas as pd
import numpy as np
from skbio.diversity import beta_diversity

In [ ]:
abundance = pd.read_csv('../../goll_et.al_2020_IBS_FMT/processed/abundance_normalized_table.csv')
rel_abundance = abundance.copy()
column_sums = rel_abundance.drop('taxon_id', axis=1).sum(axis=0)
rel_abundance.iloc[:, 1:] = rel_abundance.iloc[:, 1:].div(column_sums, axis=1)
# compute sample-level bray-curtis distance
rel_abundance_T = rel_abundance.set_index('taxon_id').transpose()
rel_abundance_T.index.name = 'sample_id'
bray_curtis_distance = beta_diversity(
    metric='braycurtis',
    counts=rel_abundance_T.values,
    ids=rel_abundance_T.index
)
bray_curtis_distance = pd.DataFrame(
    bray_curtis_distance.data,
    index=bray_curtis_distance.ids,
    columns=bray_curtis_distance.ids
)
bray_curtis_distance.to_csv('../data/bray_curtis_distance.csv', index=True)
DonorRecipientMapping = pd.read_csv('../../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')[['recipient', 'clinical_response']]
recipient = (
    DonorRecipientMapping
    .assign(
        recipient_t0=lambda x: x['recipient'].str.split('_').str[0],
        recipient_t6=lambda x: x['recipient'].str.split('_').str[0].str.replace('-0', '-6')
    )
    .drop(columns=['recipient'])
    .melt(id_vars=['clinical_response'], var_name='recipient', value_name='sample_id')
    .drop(columns=['recipient'])
)
taxonomy = pd.read_csv('../../goll_et.al_2020_IBS_FMT/processed/full_taxonomy.csv')

rel_abundance_phylum = (
    rel_abundance
    .merge(taxonomy, on='taxon_id', how='left')
    .drop(columns=['strain'])
    .melt(id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'], var_name='sample_id', value_name='abundance')
    .groupby(['phylum', 'sample_id'], as_index=False)['abundance']
    .sum()
    .pivot_table(index='phylum', columns='sample_id', values='abundance', fill_value=0)
    .transpose()
    .merge(recipient, on='sample_id', how='left')
    .assign(clinical_response=lambda x: x['clinical_response'].fillna('donor'))
    .assign(time=lambda x: np.where(
        x['sample_id'].str.contains('-6'),
        'T6',
        'T0'
    ))
)
rel_abundance_phylum['time'] = rel_abundance_phylum.apply(lambda x: 'donor' if x['sample_id'].startswith('D') else x['time'], axis=1)
rel_abundance_phylum.to_csv('../data/supp_4.csv')
rel_abundance_phylum